# Spark Expectations - ANSI Mode Testing (TLE-1230)

This notebook verifies Spark Expectations compatibility with **ANSI mode** (`spark.sql.ansi.enabled=true`), which is the default on **Databricks Serverless Compute**.

## Background

Under ANSI mode, operations like `CAST('abc' AS INT)` raise `CAST_INVALID_INPUT` instead of returning `NULL`. Spark Expectations uses `try_cast` internally to handle this safely.

## Usage

- **Databricks Serverless**: Attach to Serverless Compute (ANSI is enabled by default) and run all cells.
- **Local / Standard Compute**: Run all cells; the notebook will temporarily enable ANSI mode for testing.

## 1. Setup and ANSI Mode Detection

In [ ]:
from spark_expectations.core import get_spark_session
from spark_expectations.core.context import SparkExpectationsContext

spark = get_spark_session()

# Check current ANSI setting (Serverless typically has this as 'true')
ansi_setting = spark.conf.get("spark.sql.ansi.enabled", "false")
print(f"Current spark.sql.ansi.enabled: {ansi_setting}")

# Create context and verify ANSI detection
context = SparkExpectationsContext(product_id="ansi_test_product", spark=spark)
print(f"Context get_ansi_enabled: {context.get_ansi_enabled}")
print(f"Type: {type(context.get_ansi_enabled).__name__}")

assert isinstance(context.get_ansi_enabled, bool), "get_ansi_enabled must return bool"

## 2. try_cast vs CAST Behavior

In [ ]:
# try_cast returns NULL for invalid input (ANSI-safe)
print("try_cast('' AS INT):")
spark.sql("SELECT try_cast('' AS INT) AS result").show()

print("try_cast('abc' AS INT):")
spark.sql("SELECT try_cast('abc' AS INT) AS result").show()

print("try_cast('123' AS INT):")
spark.sql("SELECT try_cast('123' AS INT) AS result").show()

In [ ]:
# Under ANSI mode, CAST would fail on invalid input
# This cell demonstrates the difference (run only if ANSI is enabled)
if context.get_ansi_enabled:
    print("ANSI mode is ON. CAST('' AS INT) would raise CAST_INVALID_INPUT.")
    print("Spark Expectations uses try_cast to avoid this.")
else:
    print("ANSI mode is OFF. Running CAST for comparison:")
    spark.sql("SELECT CAST('' AS INT) AS result").show()

## 3. Row DQ Rules with ANSI Mode

In [ ]:
from spark_expectations.utils.actions import SparkExpectationsActions

# Sample data
df = spark.createDataFrame([
    {"row_id": 0, "col1": 1, "col2": "a"},
    {"row_id": 1, "col1": 2, "col2": "b"},
    {"row_id": 2, "col1": 3, "col2": "c"},
])

expectations = {
    "row_dq_rules": [
        {
            "product_id": "product_1",
            "rule_type": "row_dq",
            "rule": "col1_gt_eq_1",
            "column_name": "col1",
            "expectation": "col1 >= 1",
            "action_if_failed": "ignore",
            "table_name": "test_table",
            "tag": "validity",
            "description": "col1 >= 1",
        },
    ]
}

result_df = SparkExpectationsActions.run_dq_rules(context, df, expectations, "row_dq")
print("Row DQ result:")
result_df.show(truncate=False)

## 4. Aggregation DQ with try_cast Expression

In [ ]:
# Data with mixed valid/invalid numeric strings
df_amount = spark.createDataFrame([
    {"row_id": 0, "amount": "100"},
    {"row_id": 1, "amount": "200"},
    {"row_id": 2, "amount": ""},
    {"row_id": 3, "amount": None},
])

dq_rule = {
    "rule_type": "agg_dq",
    "rule": "sum_amount_gt_zero",
    "column_name": "amount",
    "expectation": "sum(try_cast(amount as int))>=0",
    "action_if_failed": "ignore",
    "table_name": "test_table",
    "tag": "validity",
    "enable_for_source_dq_validation": True,
    "description": "sum of amount should be non-negative",
    "product_id": "product_1",
}

result_out, result_df = SparkExpectationsActions.agg_query_dq_detailed_result(
    context, dq_rule, df_amount, []
)
print(f"Status: {result_df[9]}")
print(f"Actual value (sum): {result_df[10]}")
assert result_df[9] == "pass", "Expected pass"
assert result_df[10] == 300, "Expected sum 100+200=300 (empty/None yield NULL, excluded from sum)"

## 5. Report Generation with Edge-Case Data

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from spark_expectations.sinks.utils.report import SparkExpectationsReport

# Build detailed-stats DataFrame with edge cases (empty strings, NULLs)
schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("rule_type", StringType(), True),
    StructField("rule", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("source_expectations", StringType(), True),
    StructField("tag", StringType(), True),
    StructField("description", StringType(), True),
    StructField("source_dq_status", StringType(), True),
    StructField("source_dq_actual_outcome", StringType(), True),
    StructField("source_dq_expected_outcome", StringType(), True),
    StructField("source_dq_actual_row_count", StringType(), True),
    StructField("source_dq_error_row_count", StringType(), True),
    StructField("source_dq_row_count", StringType(), True),
    StructField("source_dq_start_time", StringType(), True),
    StructField("source_dq_end_time", StringType(), True),
    StructField("target_expectations", StringType(), True),
    StructField("target_dq_status", StringType(), True),
    StructField("target_dq_actual_outcome", StringType(), True),
    StructField("target_dq_expected_outcome", StringType(), True),
    StructField("target_dq_actual_row_count", StringType(), True),
    StructField("target_dq_error_row_count", StringType(), True),
    StructField("target_dq_row_count", StringType(), True),
    StructField("target_dq_start_time", StringType(), True),
    StructField("target_dq_end_time", StringType(), True),
    StructField("dq_date", StringType(), True),
    StructField("dq_time", StringType(), True),
    StructField("dq_job_metadata_info", StringType(), True),
])

data = [
    (
        "run_1", "product", "test_table", "query_dq", "rule_1", "col1", "SELECT 1",
        "validity", "desc", "pass", "0", "0", "100", "0", "100",
        "2025-02-11 00:07:39", "2025-02-11 00:07:40",
        None, None, None, None, None, None, None, None, None,
        "2025-02-10", "2025-02-11 00:07:46",
        "{'job': 'test_job', 'Region': 'NA', 'Snapshot': '2024-04-15', 'data_object_name': 'customer_order'}",
    ),
    # Edge case: empty string row counts (must not raise CAST_INVALID_INPUT)
    (
        "run_2", "product", "test_table", "query_dq", "rule_2", "col2", "SELECT 1",
        "validity", "desc", "fail", "0", "0", "", "0", "",
        "2025-02-11 00:07:39", "2025-02-11 00:07:40",
        None, None, None, None, None, None, None, None, None,
        "2025-02-10", "2025-02-11 00:07:46",
        "{'job': 'test_job', 'Region': 'NA', 'Snapshot': '2024-04-15', 'data_object_name': 'customer_order'}",
    ),
]

df_detailed = spark.createDataFrame(data, schema)

# Minimal custom detailed data (required for report)
custom_schema = "run_id string, product_id string, table_name string, rule string, column_name string, alias string, dq_type string, source_output string, target_output string, dq_time string"
custom_data = [
    ("run_1", "product", "test_table", "rule_1", "col1", "source_f1", "_source_dq", "{'source_f1=[]}", "NULL", "2025-02-11 00:07:46"),
    ("run_2", "product", "test_table", "rule_2", "col2", "source_f2", "_source_dq", "{'source_f2=[]}", "NULL", "2025-02-11 00:07:46"),
]
df_custom = spark.createDataFrame(custom_data, custom_schema)

context.set_stats_detailed_dataframe(df_detailed)
context.set_custom_detailed_dataframe(df_custom)
report = SparkExpectationsReport(context)
test_result, test_df = report.dq_obs_report_data_insert()

print(f"Report success: {test_result}")
test_df.show(truncate=False)
assert test_result is True, "Report generation must succeed with ANSI mode"

## 6. Summary

In [ ]:
print("=" * 60)
print("ANSI Mode Testing Complete (TLE-1230)")
print("=" * 60)
print("\u2713 ANSI mode detection")
print("\u2713 try_cast vs CAST behavior")
print("\u2713 Row DQ rules with ANSI")
print("\u2713 Agg DQ with try_cast expression")
print("\u2713 Report generation with edge-case data")
print("=" * 60)